In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.sql.functions import explode, col

schema = StructType([
    StructField("hr", StringType(), True),
    StructField("idlinha", StringType(), True),
    StructField("sentido", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True),
    StructField("velocidade", IntegerType(), True),
    StructField("hora_onibus", StringType(), True),
])


In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [4]:
routes_df = spark.read.option("header", True).csv("s3a://gtfs/routes.txt")
stops_df = spark.read.option("header", True).csv("s3a://gtfs/stops.txt")
trips_df = spark.read.option("header", True).csv("s3a://gtfs/trips.txt")
shapes_df = spark.read.option("header", True).csv("s3a://gtfs/shapes.txt")

In [6]:
silver_path = "s3a://silver/posicoes"
df_silver = (
    spark.readStream
    .schema(schema)
    .format("parquet")
    .load(silver_path)
)

In [7]:
df_silver = df_silver.withColumnRenamed("idlinha", "route_id")

routes_df = routes_df.withColumnRenamed("route_id", "route_id")
trips_df = trips_df.withColumnRenamed("route_id", "route_id")

In [8]:
from pyspark.sql.functions import broadcast

df_gold = (
    df_silver
    .join(broadcast(routes_df.select("route_id", "route_short_name", "route_long_name", "route_type")), on="route_id", how="left")
    .join(broadcast(trips_df.select("route_id", "trip_id", "shape_id")), on="route_id", how="left")
    .join(broadcast(shapes_df.select("shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence")), on="shape_id", how="left")
)

In [9]:
query = (
    df_gold.writeStream
    .format("parquet")
    .option("checkpointLocation", "s3a://gold/checkpoints/posicoes_gold")
    .option("path", "s3a://gold/posicoes_gold")
    .outputMode("append")
    .start()
)

query.awaitTermination()

StreamingQueryException: Query [id = 13793ba2-ea00-41ef-81ef-4fa8f7c03083, runId = 4d3eb96f-540c-4ec1-a084-e77ac5ad147e] terminated with exception: Job aborted.